In [1]:
import pandas as pd
import os

# Build the path relative to the notebook's location
data_path = os.path.join("..", "Data", "perenual_data.csv")

df = pd.read_csv(data_path)

for col in df.columns:
    uniques = df[col].unique()
    print(f"{col}: {len(uniques)}")
    print(f"{col}: {uniques[:10]}")

Unnamed: 0: 3000
Unnamed: 0: [0 1 2 3 4 5 6 7 8 9]
id: 3000
id: [ 1  2  3  4  5  6  7  8  9 10]
common_name: 1548
common_name: ['European Silver Fir' 'Pyramidalis Silver Fir' 'White Fir'
 'Candicans White Fir' 'Fraser Fir' 'Golden Korean Fir' 'Alpine Fir'
 'Blue Spanish Fir' 'Noble Fir' 'Johin Japanese Maple']
scientific_name: 3000
scientific_name: ["['Abies alba']" '["Abies alba \'Pyramidalis\'"]' "['Abies concolor']"
 '["Abies concolor \'Candicans\'"]' "['Abies fraseri']"
 '["Abies koreana \'Aurea\'"]' "['Abies lasiocarpa']"
 '["Abies pinsapo \'Glauca\'"]' "['Abies procera']" '["Acer \'Johin\'"]']
other_name: 281
other_name: ["['Common Silver Fir']" '[]'
 "['Silver Fir', 'Concolor Fir', 'Colorado Fir']" "['Southern Fir']"
 "['Subalpine Fir', 'Rocky Mountain Fir']" "['Glaucous Spanish Fir']"
 "['Red Fir', 'White Fir']" "['Red Full Moon Maple']"
 '["Father David\'s Maple", "Pere David\'s Maple"]'
 "['fernleaf full moon maple']"]
family: 151
family: ['Pinaceae' nan 'Sapindaceae' 'Fabace

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

OPEN_AI_KEY = os.getenv("OPEN_AI_KEY")

In [14]:
# type, dimensions, watering, plant_anatomy, sunlight, maintenance, seeds, soil, description (without plant name)
import re
from openai import OpenAI

client = OpenAI(api_key=OPEN_AI_KEY)

def generate_new_description(row):
    """
    row: a pandas Series with columns:
    type, dimensions, watering, plant_anatomy, sunlight, maintenance, seeds, soil, description, name
    """

    prompt = f"""
    You are an expert horticulturist. Write an objective, factual description of a plant
    using the structured attributes provided.
    
    REQUIREMENTS:
    - Do NOT mention the plant name.
    - Do NOT mention the scientific name.
    - Do NOT copy or paraphrase the original description.
    - Use clear and neutral language with no marketing or subjective qualifiers.
    - Focus on characteristics, growth conditions, and functional traits.
    - Write 2–4 concise sentences.
    
    Attributes:
    - Type: {row['type']}
    - Dimensions: {row['dimensions']}
    - Watering: {row['watering']}
    - Plant Anatomy: {row['plant_anatomy']}
    - Sunlight: {row['sunlight']}
    - Maintenance: {row['maintenance']}
    - Seeds: {row['seeds']}
    - Soil: {row['soil']}
    
    Original description (this contains common and scientific plant name, both of which should be removed):
    \"\"\"{row['description']}\"\"\"\n
    Write a new description now:
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=150,
    )

    return response.choices[0].message.content.strip()

In [15]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_row(row):
    name = row["common_name"]
    new_desc = generate_new_description(row)
    return {"name": name, "generated_description": new_desc}

generated_rows = []
futures = []

with ThreadPoolExecutor(max_workers=8) as executor:
    for idx, row in df.iterrows():
        futures.append(executor.submit(process_row, row))

    for future in as_completed(futures):
        generated_rows.append(future.result())

generated_df = pd.DataFrame(generated_rows)

In [16]:
print(generated_df.head())

                     name                              generated_description
0     European Silver Fir  This tree typically reaches a height of 60 fee...
1     Candicans White Fir  This tree reaches a height of 30 feet, charact...
2              Fraser Fir  This tree typically reaches a height of 35 fee...
3              Alpine Fir  This tree typically reaches a height of approx...
4  Pyramidalis Silver Fir  This coniferous tree typically reaches a heigh...


In [21]:
generated_df.iloc[139][1]

/var/folders/rg/9cm46k1n0tg4xzj9pqxqrx_c0000gn/T/ipykernel_1551/371096211.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  generated_df.iloc[139][1]


'This tree typically reaches a height of approximately 6 feet, making it suitable for smaller landscapes. It thrives with average watering requirements and can grow in full sun to part shade conditions. The foliage exhibits a color transition from blush-pink in spring to green, followed by vibrant fall colors of orange, gold, and red. It does not produce seeds and is characterized by a graceful form that contributes to its ornamental value.'

In [22]:
output_path = os.path.join("..", "Data", "generated_descriptions.csv")
generated_df.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: ../Data/generated_descriptions.csv
